# 34 — Caption Gemini 3.8 qua cổng Trammay (6 worker Kaggle)

Sinh lại caption cho **176.707 keyframe** bằng prompt `v5_bao`, gọi qua cổng tương thích OpenAI của Đạt. CPU-only — không cần GPU, không nạp model nào.

**Vì sao đổi khỏi Qwen2.5-VL:** đo 20-09 trên 200 keyframe, cùng thước `caption_audit`. Hàng `≥3 gốc động từ` (hàng phân biệt tốt nhất): Qwen+aic **5,0%** · Gemini+aic 10,0% · Gemini+sota_v2 19,5% · **Gemini+v5_bao 49,7%**. McNemar so với cả ba: **p = 0,0000**. Và v5_bao là bản duy nhất mà *mật độ* động từ cũng cao nhất (1,32/100 token) — nó không thắng chỉ nhờ viết dài.

**Quy trình:** `PREFLIGHT` một video để kiểm đường ống → 6 người chạy `SHARD` riêng → một người chạy `AGGREGATE`.

## Trước khi Run All

1. **Accelerator: None (CPU)** — notebook không nạp model nào.

2. **Internet: ON** — cần cho `git clone`, `pip install modal` và gọi API.

3. **Add-ons → Secrets**, đủ **bốn** tên (mỗi tài khoản Kaggle tự thêm):

   | secret | dùng làm gì |
   |---|---|
   | `GITHUB_TOKEN` | clone repo — repo này **private**, thiếu token là notebook TREO ở bước clone chứ không báo lỗi |
   | `TRAMMAY_API_KEY` | gọi cổng caption |
   | `MODAL_TOKEN_ID` | đẩy kết quả lên Volume |
   | `MODAL_TOKEN_SECRET` | ditto |

4. **Gắn dataset keyframes + map-keyframes.** Ô "Chia việc" tự quét `/kaggle/input/**` rồi in số video tìm được — **không ra 873 thì DỪNG**, đừng chạy `SHARD`.

## ⚠ Ba chỗ đã trả giá, đừng gỡ

**1. Khoá là `frame_idx`, KHÔNG phải `n`.** Notebook thử nghiệm trước đặt khoá `frame_{n:03d}`. `n` là số thứ tự keyframe *và* tên file ảnh; `frame_idx` là khung thật trong video, và `map-keyframes/{video}.csv` là chỗ duy nhất nối hai số đó. Nhầm chúng là cách dễ nhất để caption gắn lên khung khác mà mọi thứ trông vẫn đúng.

**2. API key đọc từ Kaggle Secrets, KHÔNG viết cứng.** Notebook tham chiếu từng có key nguyên văn trong cell 2.

**3. `MAX_WORKERS` 28 là con số của Kaggle, không phải của mọi máy.** Đo 21-09 trên máy Lead: 16 luồng → 800/800 lượt sạch; 30 luồng → `EOF occurred in violation of protocol` hàng loạt sau ~100 ảnh, tức bị chặn ở tầng TLS. Nếu Kaggle cũng đứt thì hạ xuống 16 rồi chạy lại — phần đã sinh được giữ nguyên.


In [ ]:
from pathlib import Path

# ── ai chạy, chạy phạm vi nào ──────────────────────────────────────────────
RUNNER = "w1"                     # w1..w6
RUNNERS = ("w1", "w2", "w3", "w4", "w5", "w6")
RUN_SCOPE = "PREFLIGHT"           # PREFLIGHT | SHARD | AGGREGATE
CONFIRM_SHARD = ""                # SHARD: đặt đúng bằng RUNNER
CONFIRM_AGGREGATE = False         # AGGREGATE: True sau khi cả 6 shard COMPLETE

# ── cổng API ───────────────────────────────────────────────────────────────
# Kaggle Secrets: Add-ons -> Secrets -> TRAMMAY_API_KEY
TRAMMAY_SECRET = "TRAMMAY_API_KEY"
MODEL_NAME = "ag/gemini-3.8-flash"
PROMPT_VARIANT = "v5_bao"
MAX_TOKENS = 768                  # bản gốc 1258 -> 8,5% timeout; xem markdown
MAX_WORKERS = 28
REQUEST_TIMEOUT = 180
MAX_RETRIES = 4

# ── đường dẫn ──────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/triet/structure"
REPO_DIR = Path("/kaggle/working/multimodal-fusion-engine")
WORK_DIR = Path("/kaggle/working/caption-gemini")
KAGGLE_INPUT = Path("/kaggle/input")

OUTPUT_ROOT = "/research/caption-gemini-trammay-v1"
RESEARCH_VOLUME = "keyframe-augmentation-research"

EXPECTED_VIDEOS = 873
EXPECTED_FRAMES = 176707          # sau khi gộp 614 cặp trùng frame_idx

if RUNNER not in RUNNERS:
    raise ValueError(f"RUNNER phải thuộc {RUNNERS}")
if RUN_SCOPE not in {"PREFLIGHT", "SHARD", "AGGREGATE"}:
    raise ValueError("RUN_SCOPE phải là PREFLIGHT, SHARD hoặc AGGREGATE")
if RUN_SCOPE == "SHARD" and CONFIRM_SHARD != RUNNER:
    raise RuntimeError("SHARD bị khoá; đặt CONFIRM_SHARD đúng bằng RUNNER")
if RUN_SCOPE == "AGGREGATE" and not CONFIRM_AGGREGATE:
    raise RuntimeError("AGGREGATE bị khoá; đặt CONFIRM_AGGREGATE = True")

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUNNER={RUNNER} · RUN_SCOPE={RUN_SCOPE} · model={MODEL_NAME} · prompt={PROMPT_VARIANT}")

In [ ]:
import subprocess, sys, os

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

# Repo là PRIVATE — `git clone` trần sẽ hỏi mật khẩu rồi TREO notebook, không
# phải báo lỗi. `GIT_ASKPASS` trả token qua stdin, `GIT_TERMINAL_PROMPT=0` biến
# mọi lần hỏi còn sót thành lỗi ngay thay vì treo. Cùng cách nb33 đã dùng.
if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    askpass = WORK_DIR / "git-askpass.sh"
    askpass.write_text(
        '#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; '
        '*) printf %s "$GITHUB_TOKEN" ;; esac\n',
        encoding="utf-8",
    )
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update({"GITHUB_TOKEN": secrets.get_secret("GITHUB_TOKEN"),
                "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL,
                    str(REPO_DIR)], check=True, env=env)
sys.path.insert(0, str(REPO_DIR / "src"))

os.environ["TRAMMAY_API_KEY"] = secrets.get_secret(TRAMMAY_SECRET)
for name in ("MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET"):
    os.environ[name] = secrets.get_secret(name)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modal"], check=True)

from mfe.contracts.ids import build_keyframe_id
from mfe.offline.caption_prompt import PROMPT_VARIANTS
from mfe.offline.transport import ModalVolumeTransport
from mfe.online.trammay import TrammayClient, TrammayError, available

PROMPT, _mac_dinh = PROMPT_VARIANTS[PROMPT_VARIANT]
ok, vi_sao = available()
if not ok:
    raise RuntimeError(vi_sao)

RESEARCH = ModalVolumeTransport(volume=RESEARCH_VOLUME, local_root=str(WORK_DIR / "dl"))
client = TrammayClient.from_env(timeout=REQUEST_TIMEOUT, max_retries=MAX_RETRIES)
client = client.__class__(**{**client.__dict__, "model": MODEL_NAME})
print(f"prompt {PROMPT_VARIANT}: {len(PROMPT)} ký tự · client sẵn sàng")

## Chia việc — theo VIDEO, không theo keyframe

Chia theo video để mỗi worker tự đủ: một video là một đơn vị marker, và hai worker không bao giờ ghi cùng một file. Phép chia là **xác định** (băm tên video), nên chạy lại ở máy khác vẫn ra đúng cùng phần việc — không cần bảng phân công lưu ở đâu cả.


In [ ]:
import csv, hashlib, json

def _tim_map_csv() -> dict:
    """`{video_id: đường dẫn CSV}` — quét mọi dataset đã gắn vào Kaggle."""
    ra = {}
    for p in KAGGLE_INPUT.rglob("map-keyframes/*.csv"):
        ra.setdefault(p.stem, p)
    return ra

def _tim_thu_muc_anh() -> dict:
    """`{video_id: thư mục ảnh}` — keyframes chia theo lô (Keyframes_L21/...)."""
    ra = {}
    for p in KAGGLE_INPUT.rglob("keyframes/*"):
        if p.is_dir() and p.name.count("_") == 1:
            ra.setdefault(p.name, p)
    return ra

MAP_CSV = _tim_map_csv()
ANH_DIR = _tim_thu_muc_anh()
print(f"map-keyframes: {len(MAP_CSV)} video · thư mục ảnh: {len(ANH_DIR)} video")
if len(MAP_CSV) != EXPECTED_VIDEOS:
    print(f"⚠ mong {EXPECTED_VIDEOS} video, thấy {len(MAP_CSV)} — kiểm dataset đã gắn đủ chưa")

def shard_cua(video_id: str) -> str:
    """Băm xác định -> tên worker. Đổi hàm này là đổi toàn bộ phân công."""
    h = hashlib.sha256(video_id.encode()).hexdigest()
    return RUNNERS[int(h[:8], 16) % len(RUNNERS)]

PHAN_CONG = {r: sorted(v for v in MAP_CSV if shard_cua(v) == r) for r in RUNNERS}
for r in RUNNERS:
    print(f"  {r}: {len(PHAN_CONG[r]):>3} video")
CUA_TOI = PHAN_CONG[RUNNER] if RUN_SCOPE != "PREFLIGHT" else PHAN_CONG[RUNNER][:1]
print(f"\n{RUNNER} xử lý {len(CUA_TOI)} video trong lượt này")

## Đọc khung — `frame_idx` lấy từ CSV, ảnh lấy theo `n`

Hai số, hai vai, và chúng **không** thay nhau được:

- **`n`** đặt tên file ảnh (`001.jpg`) — dùng để MỞ ảnh;

- **`frame_idx`** là khung thật trong video — dùng để ĐẶT KHOÁ.

`build_keyframe_id(video_id, frame_idx)` là đường duy nhất dựng khoá. Đừng tự f-string.

**614 cặp keyframe của batch 1 dùng chung một `frame_idx`** (artefact làm tròn của BTC, 0,346%). Gộp phải **giữ bản đầu tiên** để ra đúng 176.707 dòng — nếu không hai arm đánh địa chỉ khác nhau và hợp nhất cộng điểm của hai khung khác nhau.


In [ ]:
def khung_cua(video_id: str) -> list:
    """`[(keyframe_id, đường dẫn ảnh)]`, đã gộp giữ bản ĐẦU TIÊN khi trùng frame_idx."""
    csv_path = MAP_CSV.get(video_id)
    thu_muc = ANH_DIR.get(video_id)
    if csv_path is None or thu_muc is None:
        return []
    ra, da_thay = [], set()
    with csv_path.open(encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            frame_idx = int(row["frame_idx"])
            if frame_idx in da_thay:      # 614 cặp trùng — giữ bản đầu
                continue
            anh = thu_muc / f"{int(row['n']):03d}.jpg"
            if not anh.is_file():
                continue
            da_thay.add(frame_idx)
            ra.append((build_keyframe_id(video_id, frame_idx), anh))
    return ra

_thu = khung_cua(CUA_TOI[0])
print(f"{CUA_TOI[0]}: {len(_thu)} khung · khoá mẫu {_thu[0][0] if _thu else '(rỗng)'}")
assert not _thu or "#" in _thu[0][0], "khoá phải dạng {video}#{frame_idx:06d}"

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with Path(p).open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_json(obj) -> str:
    return hashlib.sha256(
        json.dumps(obj, sort_keys=True, ensure_ascii=False).encode()
    ).hexdigest()

def duong_remote(video_id: str) -> dict:
    return {
        "captions": f"{OUTPUT_ROOT}/parts/captions/{video_id}.json",
        "marker": f"{OUTPUT_ROOT}/markers/{RUNNER}/{video_id}.json",
    }

def marker_hop_le(video_id: str):
    """Marker chỉ tính là xong khi hash của FILE khớp hash ghi trong marker.

    Marker là điểm chốt, không phải file: một part đẩy dở vẫn tồn tại trên
    volume, nên "có file" không thể có nghĩa là "đã xong".
    """
    paths = duong_remote(video_id)
    if not RESEARCH.exists(paths["marker"]):
        return None
    local = WORK_DIR / "marker-check" / f"{video_id}.json"
    local.parent.mkdir(parents=True, exist_ok=True)
    try:
        RESEARCH.get(paths["marker"], local)
        marker = json.loads(local.read_text(encoding="utf-8"))
    except Exception:
        return None
    if marker.get("status") != "COMPLETE" or marker.get("runner") != RUNNER:
        return None
    if marker.get("prompt_variant") != PROMPT_VARIANT or marker.get("model") != MODEL_NAME:
        return None   # đổi prompt/model là phải sinh lại, không dùng lại bản cũ
    if RESEARCH.sha256(paths["captions"]) != marker.get("captions_sha256"):
        return None
    return marker

def mot_khung(item):
    kid, anh = item
    try:
        return kid, client.chat(PROMPT, images=[anh], max_tokens=MAX_TOKENS,
                                temperature=0.0)
    except TrammayError:
        # Client đã tự thử lại MAX_RETRIES lần; tới đây là hết đường cho khung
        # này. Trả None để vòng ngoài ĐẾM nó thay vì để cả video chết.
        return kid, None

def xu_ly_video(video_id: str, publish: bool) -> dict:
    khung = khung_cua(video_id)
    local = WORK_DIR / "parts" / video_id
    local.mkdir(parents=True, exist_ok=True)
    file_cap = local / "captions.json"
    caps = json.loads(file_cap.read_text(encoding="utf-8")) if file_cap.is_file() else {}
    con = [(k, a) for k, a in khung if k not in caps]

    t0, xong, hong = time.time(), 0, 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        for kid, text in ex.map(mot_khung, con):
            if text is None:
                hong += 1
                continue
            caps[kid] = text.replace("[FINAL_CAPTION]", "").strip()
            xong += 1
            if xong % 50 == 0:      # ghi dày: mất mạng thì chỉ mất <50 ảnh
                file_cap.write_text(json.dumps(caps, ensure_ascii=False), encoding="utf-8")
    file_cap.write_text(json.dumps(caps, ensure_ascii=False), encoding="utf-8")

    bao_cao = {"video_id": video_id, "khung": len(khung), "caption": len(caps),
               "hong": hong, "giay": round(time.time() - t0, 1)}
    if not publish:
        return bao_cao
    if len(caps) != len(khung):
        bao_cao["status"] = "PARTIAL"   # thiếu khung thì KHÔNG ghi marker
        return bao_cao

    remote = duong_remote(video_id)
    RESEARCH.put(file_cap, remote["captions"])
    if RESEARCH.sha256(remote["captions"]) != sha256_file(file_cap):
        raise RuntimeError(f"{video_id}: captions hash mismatch sau khi đẩy")
    marker = {"schema_version": 1, "status": "COMPLETE", "runner": RUNNER,
              "video_id": video_id, "frames": len(caps), "model": MODEL_NAME,
              "prompt_variant": PROMPT_VARIANT, "max_tokens": MAX_TOKENS,
              "captions_sha256": sha256_file(file_cap),
              "heldout_accessed": False, "btc_mutated": False,
              "completed_at_utc": datetime.now(timezone.utc).isoformat()}
    marker["marker_payload_hash"] = sha256_json(marker)
    mp = local / "marker.json"
    mp.write_text(json.dumps(marker, ensure_ascii=False), encoding="utf-8")
    RESEARCH.put(mp, remote["marker"])      # marker ghi SAU CÙNG
    if RESEARCH.sha256(remote["marker"]) != sha256_file(mp):
        raise RuntimeError(f"{video_id}: marker hash mismatch")
    bao_cao["status"] = "COMPLETE"
    return bao_cao

In [ ]:
if RUN_SCOPE in {"PREFLIGHT", "SHARD"}:
    xong_roi = ([v for v in CUA_TOI if marker_hop_le(v)]
                if RUN_SCOPE == "SHARD" else [])
    con_lai = [v for v in CUA_TOI if v not in set(xong_roi)]
    print(f"{len(xong_roi)} video đã COMPLETE · còn {len(con_lai)}")

    bao_caos, that_bai = [], []
    for i, video_id in enumerate(con_lai, 1):
        try:
            r = xu_ly_video(video_id, publish=RUN_SCOPE == "SHARD")
            bao_caos.append(r)
            print(f"[{i}/{len(con_lai)}] {video_id}: {r['caption']}/{r['khung']} "
                  f"· hỏng {r['hong']} · {r['giay']}s · {r.get('status', 'PREFLIGHT')}")
        except Exception as e:
            that_bai.append({"video_id": video_id, "loi": repr(e)})
            print(f"[{i}/{len(con_lai)}] {video_id}: THẤT BẠI {e!r}")

    tong_hong = sum(r["hong"] for r in bao_caos)
    tong_cap = sum(r["caption"] for r in bao_caos)
    print(f"\nTổng: {tong_cap:,} caption · {tong_hong} khung hỏng "
          f"({tong_hong / max(1, tong_cap + tong_hong) * 100:.1f}%)")
    if tong_hong / max(1, tong_cap + tong_hong) > 0.02:
        print("⚠ >2% hỏng — hạ MAX_WORKERS xuống 16 rồi chạy lại ô này")

In [ ]:
if RUN_SCOPE == "AGGREGATE":
    gop, trung = {}, 0
    thieu_runner = []
    for r in RUNNERS:
        for video_id in PHAN_CONG[r]:
            remote = f"{OUTPUT_ROOT}/parts/captions/{video_id}.json"
            if not RESEARCH.exists(remote):
                thieu_runner.append((r, video_id))
                continue
            local = WORK_DIR / "gop" / f"{video_id}.json"
            local.parent.mkdir(parents=True, exist_ok=True)
            RESEARCH.get(remote, local)
            for k, v in json.loads(local.read_text(encoding="utf-8")).items():
                if k in gop:
                    trung += 1     # hai worker cùng ghi một khoá = phân công hỏng
                    continue
                gop[k] = v

    print(f"gộp được {len(gop):,} caption · trùng khoá {trung} · thiếu {len(thieu_runner)} video")
    if trung:
        raise RuntimeError(f"{trung} khoá trùng — hai worker chồng lấn, KHÔNG được gộp")
    if len(gop) != EXPECTED_FRAMES:
        print(f"⚠ mong {EXPECTED_FRAMES:,}, được {len(gop):,} "
              f"(lệch {len(gop) - EXPECTED_FRAMES:+,}) — kiểm trước khi dùng")

    ra = WORK_DIR / "qwen_captions.json"   # giữ đúng tên file kho đang đọc
    ra.write_text(json.dumps(gop, ensure_ascii=False), encoding="utf-8")
    RESEARCH.put(ra, f"{OUTPUT_ROOT}/qwen_captions.json")
    manifest = {"schema_version": 1,
                "status": "COMPLETE" if len(gop) == EXPECTED_FRAMES else "PARTIAL",
                "frames": len(gop), "model": MODEL_NAME,
                "prompt_variant": PROMPT_VARIANT, "max_tokens": MAX_TOKENS,
                "missing_videos": [v for _, v in thieu_runner],
                "captions_sha256": sha256_file(ra),
                "heldout_accessed": False, "btc_mutated": False,
                "created_at_utc": datetime.now(timezone.utc).isoformat()}
    manifest["manifest_payload_hash"] = sha256_json(manifest)
    mp = WORK_DIR / "manifest.json"
    mp.write_text(json.dumps(manifest, ensure_ascii=False), encoding="utf-8")
    RESEARCH.put(mp, f"{OUTPUT_ROOT}/manifest.json")
    print(f"AGGREGATE {manifest['status']}: {len(gop):,} caption -> {OUTPUT_ROOT}")

## Sau khi AGGREGATE

1. **Đếm lại**: `qwen_captions.json` phải có đúng **176.707** khoá. Lệch là có worker chồng lấn hoặc sót video — ô AGGREGATE đã ném lỗi khi thấy khoá trùng.

2. **Chấm chất lượng** trước khi thay kho, bằng chính thước của repo:

```python
from mfe.offline.caption_audit import audit, format_audit
print(format_audit(audit((c, "") for c in gop.values()), "Gemini v5_bao"))
```
Hàng cần nhìn là **`≥3 gốc động từ`**, không phải hàng `quan hệ` có/không — hàng đó bão hoà sớm và không phân biệt được gì. Mốc để so: Qwen+aic **5,0%**, Gemini+v5_bao trên mẫu 200 ảnh **49,7%**.

3. **`bge_m3` phải nhúng lại.** `dense_vectors.npz` là nhúng của caption CŨ; giữ nguyên nó cạnh caption mới là hai arm mô tả hai thứ khác nhau dưới cùng một khoá.

4. **Leo dốc lại bảng trọng số, và XOÁ `--evidence-cache` cũ.** Vân tay của cache chỉ bắt config/queryset/topk/en-source — nó **không** bắt được việc đổi nội dung kho. Dùng lại cache cũ là leo dốc trên kết quả truy hồi của kho cũ mà vẫn in ra một bảng đẹp.

5. **`caption_qwen` giờ phải ăn truy vấn ĐÃ DỊCH** (`query_language="en"`, sửa 21-09). Đo trên 163 câu dev: truy vấn tiếng Việt trúng 27/163, bản dịch trúng **122/163**.
